# 05 - Métodos de ponto fixo
Vamos aprender sobre como usar os métodos de ponto fixo para encontrar as raízes de funções reais.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana5
```

Faça o checkout nessa nova branch:

```bash
git checkout semana5
```

Instale as bibliotecas NumPy e SciPy.

In [14]:
%pip install numpy scipy


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


<hr />

## Atividade 1
Escreva os métodos de ponto fixo dentro do arquivo $utils/algoritmos.py$.

In [15]:
import numpy as np
from scipy.optimize import approx_fprime

def pontofixo(a, g, TOL=1e-8):
    x = g(a)
    while abs(x - a) > TOL:
        a = x
        x = g(a)
    return x

def newton_raphson(a, f, TOL=1e-8, df=None):
    if df is None:
        def dfn(x):
            return approx_fprime(np.array([x]), lambda v: f(v[0]))[0]
    else:
        dfn = df
    g = lambda x: x - f(x) / dfn(x)
    return pontofixo(a, g, TOL)

def secante(a, b, f, TOL=1e-8):
    g = lambda a, b: (a * f(b) - b * f(a)) / (f(b) - f(a))
    x = g(a, b)
    while abs(x - b) > TOL:
        a, b = b, x
        x = g(a, b)
    return x

Resolver a equação $e^x = x + 2$ é equivalente a calcular os pontos fixos da função  

$$
g(x) = e^x - 2
$$

Use os métodos do ponto fixo  

$$
x^{(n+1)} = g(x^{(n)})
$$

com $x^{(0)} = -1.8$ para obter uma aproximação de uma das soluções da equação dada com 8 dígitos significativos.  

**Resposta:**  
$
x \approx -1.8414057
$

In [16]:
import numpy as np

f1 = lambda x: np.e**x - x - 2
g1 = lambda x: np.e**x - 2


def main():
    r = pontofixo(-1.8, g1)
    print(f"raiz ponto fixo = {r}")
    r = newton_raphson(-1.8, f1, df=lambda x: np.e**x - 1)
    print(f"raiz newton-raphson df = {r}")
    r = newton_raphson(-1.8, f1)
    print(f"raiz newton-raphson = {r}")
    r = secante(-1.8, -1.7, f1)
    print(f"raiz secante = {r}")


if __name__ == "__main__":
    main()


raiz ponto fixo = -1.841405660009702
raiz newton-raphson df = -1.8414056604369606
raiz newton-raphson = -1.8414056604369609
raiz secante = -1.841405660436961


## Atividade 2
Encontre a raiz positiva da função  

$$
f(x) = \cos(x) - x^2
$$  

pelos métodos do ponto fixo, inicializando-o com $x^{(0)} = 1$.  

Realize a iteração até obter estabilidade no **quinto dígito significativo**.  

**Resposta:**

$$
x \approx 0.82413 
$$

Processo iterativo:  

$$
x^{(n+1)} = x^{(n)} + \frac{\cos(x) - x^2}{\sin(x) + 2x}
$$

In [17]:
import numpy as np

f2 = lambda x: np.cos(x) - x**2
g2 = lambda x: x + (np.cos(x) - x**2) / (np.sin(x) + 2*x)

r = pontofixo(1, g2, TOL=1e-5)
print(f"raiz = {r:.5f}")


raiz = 0.82413


## Atividade 3
Aplique os métodos do ponto fixo para resolver a equação:

$$
e^{-x^2} = 2x
$$

**Resposta:**

$$
x \approx 0.4193648
$$

In [18]:
import numpy as np

f3 = lambda x: np.exp(-x**2) - 2*x
g3 = lambda x: np.exp(-x**2) / 2

r = pontofixo(0.5, g3)
print(f"raiz ponto fixo = {r}")

r = newton_raphson(
    0.5,
    f3,
    df=lambda x: -2*x*np.exp(-x**2) - 2
)
print(f"raiz newton-raphson = {r}")

r = secante(0.4, 0.5, f3)
print(f"raiz secante = {r}")


raiz ponto fixo = 0.4193648224048179
raiz newton-raphson = 0.41936482401913244
raiz secante = 0.41936482401913217


## Atividade 4
Resolva os três últimos exercícios da semana anterior usando os métodos do ponto fixo.

In [19]:
import math

k = 1.380649e-23
q = 1.602176634e-19
T = 300
IR = 1e-12
vt = k * T / q

def raiz_diodo(V, R):
    f = lambda vd: R * IR * math.expm1(vd / vt) + vd - V
    df = lambda vd: R * IR * math.exp(vd / vt) / vt + 1
    g = lambda vd: vd - f(vd) / df(vd)

    x0 = 0.7 if V > 0.7 else V
    return pontofixo(x0, g, TOL=1e-10)

casos = [
    (30, 1e3),
    (3, 1e3),
    (3, 10e3),
    (0.3, 1e3),
    (-0.3, 1e3),
    (-30, 1e3),
    (-30, 10e3),
]

print("Exercício 1 - Diodo")
for V, R in casos:
    vd = raiz_diodo(V, R)
    print(f"V = {V:g} V, R = {R/1000:g} kΩ -> vd = {vd:.6f} V")

d = 500
fmax = 50

f_cabo = lambda C: C * (math.cosh(d / (2 * C)) - 1) - fmax
df_cabo = lambda C: (
    math.cosh(d / (2 * C))
    - 1
    - (d / (2 * C)) * math.sinh(d / (2 * C))
)
g_cabo = lambda C: C - f_cabo(C) / df_cabo(C)

C = pontofixo(650, g_cabo, TOL=1e-10)

print("\nExercício 2 - Catenária")
print(f"C = {C:.4f} m")

freq = 1000
L = 0.1
R = 1000

tan_phi = 2 * math.pi * freq * L / R
phi = math.atan(tan_phi)

Id = lambda beta: (
    math.sin(beta - phi)
    + math.sin(phi) * math.exp(-beta / tan_phi)
)
dId = lambda beta: (
    math.cos(beta - phi)
    - (math.sin(phi) / tan_phi) * math.exp(-beta / tan_phi)
)
g_beta = lambda beta: beta - Id(beta) / dId(beta)

beta = pontofixo(math.radians(210), g_beta, TOL=1e-12)

print("\nExercício 3 - Retificador RL")
print(f"tan(phi) = {tan_phi:.6f}")
print(f"phi = {math.degrees(phi):.4f} graus")
print(f"beta = {math.degrees(beta):.4f} graus")


Exercício 1 - Diodo
V = 30 V, R = 1 kΩ -> vd = 0.623123 V
V = 3 V, R = 1 kΩ -> vd = 0.558810 V
V = 3 V, R = 10 kΩ -> vd = 0.499900 V
V = 0.3 V, R = 1 kΩ -> vd = 0.299891 V
V = -0.3 V, R = 1 kΩ -> vd = -0.300000 V
V = -30 V, R = 1 kΩ -> vd = -30.000000 V
V = -30 V, R = 10 kΩ -> vd = -30.000000 V

Exercício 2 - Catenária
C = 633.1622 m

Exercício 3 - Retificador RL
tan(phi) = 0.628319
phi = 32.1419 graus
beta = 212.2258 graus


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 5"
git push origin semana5
```